In [ ]:
import uuid

import httpx

from a2a.client import A2ACardResolver, ClientFactory
from a2a.types import (
    Message,
    Part,
    Role,
    TextPart
)

In [4]:
BASE_URL = "http://localhost:10001"
PUBLIC_AGENT_CARD_PATH = "/.well-known/agent.json"

In [5]:
async with httpx.AsyncClient() as httpx_client:
    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL
    )

    try:
        print(
            f"Fetching public agent card from: {BASE_URL}{PUBLIC_AGENT_CARD_PATH}"
        )
        _public_card = await resolver.get_agent_card()
        print("Fetched public agent card")
        print(_public_card.model_dump_json(indent=2))

        final_agent_card = _public_card
    
    except Exception as e:
        error_msg = f"Error fetching public agent card: {e}"
        print(error_msg)
        raise RuntimeError(error_msg)

Fetching public agent card from: http://localhost:10001/.well-known/agent.json
Fetched public agent card
{
  "additionalInterfaces": null,
  "capabilities": {
    "extensions": null,
    "pushNotifications": null,
    "stateTransitionHistory": null,
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "A agent that can check the availability of items in the warehouses and reserve them.",
  "documentationUrl": null,
  "iconUrl": null,
  "name": "warehouse_manager_agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "provider": null,
  "security": null,
  "securitySchemes": null,
  "signatures": null,
  "skills": [
    {
      "description": "Check the availability of items in the warehouses",
      "examples": [
        "what is the availability of the item 123?"
      ],
      "id": "ABC",
      "inputModes": null,
      "name": "Check Availability",
      "outputModes": null,
      "securit

In [7]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)

client = await ClientFactory.connect(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.user,
    messageId=message_id,
    parts=[Part(root=TextPart(text="What is the availability of B0CC61QC4J in all of the warehouses?"))],
)

print("Sending message")
final_response = None
async for response in client.send_message(message_payload):
    final_response = response

print("Response:")
for artifact in final_response[0].artifacts:
    for part in artifact.parts:
        print(part.root.text)



A2AClient initialized
Sending message
Response:
The product B0CC61QC4J is available in all of the warehouses. Here is the availability by warehouse:

- Berlin Distribution Center (Berlin, Germany): 72 units
- Lyon Regional Warehouse (Lyon, France): 21 units
- Munich Logistics Hub (Munich, Germany): 33 units
- Paris Central Depot (Paris, France): 60 units
- Marseille Mediterranean Hub (Marseille, France): 77 units
- Hamburg North Warehouse (Hamburg, Germany): 88 units

If you need, I can help you reserve this product from any of these warehouses.
